In [1]:
# Importer des bibliothèques standard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import math
import seaborn as sns
import re

# Importer des bibliothèques pour le prétraitement des données
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score

# Importation de bibliothèques pour les mesures de similarité et de distance
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Importation de bibliothèques pour l'analyse de texte
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Importer les bibliothèques pour Colab et la gestion des fichiers
from google.colab import drive
from google.colab import files

# Outils divers
import sklearn

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Charger les fichiers
interactions = pd.read_csv('https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/interactions_train.csv')
items = pd.read_csv("https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/items.csv")

# Afficher les premières lignes de chaque ensemble de données
display(interactions.head())
display(items.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


In [3]:
# Renommer les colonnes du dataset interactions
interactions.rename(columns={
    'u': 'user_id',        # Colonne utilisateur
    'i': 'item_id',        # Colonne identifiant du livre
    't': 'timestamp'       # Colonne horodatage
}, inplace=True)

# Renommer les colonnes du dataset items
items.rename(columns={
    'Title': 'title',          # Titre du livre
    'Author': 'author',        # Auteur
    'ISBN Valid': 'isbn',      # ISBN
    'Publisher': 'publisher',  # Éditeur
    'Subjects': 'subjects',    # Catégories/thèmes
    'i': 'item_id'             # Identifiant du livre
}, inplace=True)

# Vérifier les colonnes après renommage
print("Colonnes interactions:", interactions.columns)
print("Colonnes items:", items.columns)

Colonnes interactions: Index(['user_id', 'item_id', 'timestamp'], dtype='object')
Colonnes items: Index(['title', 'author', 'isbn', 'publisher', 'subjects', 'item_id'], dtype='object')


This code renames the columns of the interactions and items datasets to improve their readability and consistency. The columns in the interactions dataset are renamed with explicit names such as user_id, item_id, and timestamp. Similarly, the columns in the items dataset are standardized to lowercase for better clarity, with names like title, author, and item_id. Finally, the modified columns are displayed to verify that the changes have been correctly applied, making the data easier to work with in subsequent steps.

In [4]:
# Fonction pour nettoyer les données textuelles dans le DataFrame items
def clean_text(text):
    if pd.isna(text):
        return ""
    # Retirer les caractères spéciaux et normaliser les espaces
    text = re.sub(r'[^\w\s]', '', text)
    # Convertir le texte en minuscules pour uniformiser
    text = text.lower()
    # Supprimer les espaces inutiles
    text = " ".join(text.split())
    return text

items['title'] = items['title'].apply(clean_text)
items['author'] = items['author'].apply(clean_text)
items['isbn'] = items['isbn'].apply(clean_text)
items['subjects'] = items['subjects'].apply(clean_text)
items['publisher'] = items['publisher'].apply(clean_text)

This code performs text cleaning for the items DataFrame.

A function clean_text is defined to clean each text value:

1. If the value is null (NaN), it returns an empty string.
2. It removes special characters and extra spaces using a regular expression.
3. It converts the text to lowercase for uniformity.
4. It removes extra spaces by splitting the text into words and rejoining them with a single space.

This function is then applied to specific text columns in the items DataFrame: title, author, isbn, subjects, and publisher. Each column is updated with its cleaned version using the .apply(clean_text) method. This cleaning process ensures that the data is consistent and ready for further analysis or modeling steps.

In [5]:
#Tri des données par utilisateur et par timestamp
interactions = interactions.sort_values(["user_id", "timestamp"])

#Calcul du rang proportionnel par utilisateur
interactions["pct_rank"] = interactions.groupby("user_id")["timestamp"].rank(pct=True, method="dense")
interactions.reset_index(inplace=True, drop=True)

#Calcul de l'atténuation temporelle
time_decay = np.exp(-(interactions["timestamp"].max() - interactions["timestamp"]) / (365 * 24 * 60 * 60))
interactions["time_decay"] = time_decay

n_users = interactions["user_id"].max() + 1
n_items = interactions["item_id"].max() + 1
print("Nombre d'utilisateurs:", n_users)
print("Nombre d'items:", n_items)


Nombre d'utilisateurs: 7838
Nombre d'items: 15291


This code prepares the interactions DataFrame for further analysis or modeling. First, the interactions are sorted by user (user_id) and chronologically (timestamp), ensuring proper temporal organization for each user. Next, a new column, pct_rank, is calculated for each interaction, representing its proportional rank within a user's interaction history. This allows for a normalized representation of the temporal position of interactions.

Another column, time_decay, is added to model temporal decay using an exponentially decreasing function. This approach assigns higher importance to recent interactions while reducing the weight of older ones.

Finally, the total number of unique users (n_users) and unique items (n_items) is computed and displayed, providing an overview of the dataset’s dimensions. These transformations make the data more relevant and suitable for time-sensitive analyses or temporal weighting.

In [6]:
# Définition de la fonction pour créer la matrice utilisateur-item
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["user_id"].values, data["item_id"].values] = data["time_decay"].values
    return data_matrix

# Créer la matrice utilisateur-item avec la pondération temporelle
train_data_matrix = create_data_matrix(interactions, n_users, n_items)

# Calcul de la similarité entre items
item_similarity = cosine_similarity(train_data_matrix.T)

# Calcul de la similarité entre utilisateurs
user_similarity = cosine_similarity(train_data_matrix)


# Définition de la fonction pour prédire avec la similarité des items
def item_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

# Générer les prédictions basées sur la similarité des items
item_prediction = item_predict(train_data_matrix, item_similarity)


# Définition de la fonction pour prédire avec la similarité des utilisateurs
def user_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

# Générer les prédictions basées sur la similarité des utilisateurs
user_prediction = user_predict(train_data_matrix, user_similarity)

This code implements the steps necessary to build a user-item matrix, compute similarities between users and items, and generate predictions based on these similarities. First, a function called create_data_matrix is defined to create a user-item matrix, where each cell represents an interaction weighted by a temporal decay factor (time_decay). This matrix is then used to compute two types of similarities: item similarity and user similarity. The similarity is calculated using the cosine_similarity function, which measures the proximity between two vectors.

Once these similarities are computed, two distinct functions are defined to generate predictions. The item_predict function uses item similarity to predict potential interactions between users and items, while the user_predict function does the same using user similarity. These predictions are normalized to ensure a balanced distribution of scores, and the results are stored in two variables: item_prediction (for predictions based on items) and user_prediction (for predictions based on users).

In [7]:
# Charger un modèle pré-entraîné
model = SentenceTransformer('all-mpnet-base-v2') #--> 0.1698 (x=4, y=3, z=3)

#TESTS
#model = SentenceTransformer('all-distilroberta-v1') --> 0.1695
#model = SentenceTransformer('multi-qa-mpnet-base-dot-v1') --> 0.1696
#model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1') --> 0.1698 (x=4, y=3, z=3)
#model = SentenceTransformer('bert-large-uncased') #--> 0.1698 (x=4, y=3, z=3)

# Create 'combined_text' column by concatenating relevant column
items['combined_text'] = items['title'].astype(str) + ' ' + items['author'].astype(str) + ' ' + items['subjects'].astype(str)

# Créer les embeddings des items à partir de la colonne combinée
combined_item_embeddings = model.encode(items['combined_text'].tolist(), show_progress_bar=True)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/478 [00:00<?, ?it/s]


This code uses a pre-trained model from the SentenceTransformer library to generate semantic embeddings from the textual information of the items. The key textual information from the columns title, author, and subjects is combined into a new column called combined_text. This step consolidates all the important information into a single text string for each item.

The pre-trained model then encodes these texts into numerical vectors (embeddings), which capture the semantic relationships between words and phrases.

In [8]:
# Calculer la similarité entre les items en utilisant les embeddings textuels
text_similarity = cosine_similarity(combined_item_embeddings)

# Prédiction des items recommandés avec la similarité combinée
def item_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

item_prediction_combined = item_predict(train_data_matrix, text_similarity)


This code uses the textual embeddings of items to calculate a similarity matrix between them, based on cosine similarity. This matrix (text_similarity) measures how semantically similar the items are to each other.

Next, the item_predict function generates recommendations by combining this similarity with the existing interactions between users and items. It calculates predictions by multiplying the similarity matrix with the interactions matrix and then normalizing the scores to ensure consistency.

Finally, the predictions based on this textual similarity are stored in the variable item_prediction_combined, allowing recommendations of items that are semantically similar, even in the absence of direct interactions.

In [9]:
# Poids des différentes composantes du modèle hybride
x = 0.4
y = 0.3
z = 1 - x - y

model_hybrid = x * user_prediction + y * item_prediction + z * item_prediction_combined


This code defines a hybrid model by combining three types of predictions: those based on user similarity (user_prediction), those based on item similarity (item_prediction), and those derived from the combined textual similarity of items (item_prediction_combined). Each type of prediction is weighted by a specific coefficient (x, y, z) that determines its importance in the final combination. The weights were adjusted through trial and error, testing different values for x, y, and z to find the optimal combination that maximizes prediction performance.

In [10]:
# Générer les 10 meilleures recommandations avec une logique unifiée
recommendations = []
for user_id in range(model_hybrid.shape[0]):
    top_10 = np.argsort(model_hybrid[user_id, :])[-10:][::-1]
    recommendations.append(" ".join(map(str, top_10)))

# Créer un DataFrame avec les recommandations
import pandas as pd  # Importation de pandas pour manipuler les DataFrames
submission_df = pd.DataFrame({
    "user_id": range(model_hybrid.shape[0]),
    "recommendation": recommendations
})

# Sauvegarder le fichier CSV dans l'environnement Colab
submission_file = "predictions_submission.csv"
submission_df.to_csv(submission_file, index=False)

# Télécharger le fichier localement
from google.colab import files
files.download(submission_file)

# Vérifier le résultat
print(submission_df.head(10))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   user_id                                     recommendation
0        0                         21 24 1 20 17 23 22 7 2 13
1        1                      39 38 31 37 33 36 35 32 30 34
2        2                      94 92 80 76 79 77 50 57 93 82
3        3            157 132 145 140 162 155 151 166 168 172
4        4            192 204 203 205 202 195 200 207 206 197
5        5            221 223 222 224 219 212 218 220 217 216
6        6   230 229 231 232 3862 13950 2829 7568 13988 10867
7        7            246 240 250 249 245 247 242 243 236 248
8        8  258 257 256 14986 4385 10616 14417 14483 5920 873
9        9    264 262 261 263 1556 4381 13709 4023 1263 11774



This code generates the top 10 recommendations for each user based on the hybrid model scores, organizes them into a table, and saves them in a CSV file.

For each user, the item scores are sorted to select the top 10. These items are formatted into a string and added to a list of recommendations.

The recommendations are then placed into a table (DataFrame) with two columns: user_id (user identifier) and recommendation (the top 10 recommended items). This table is exported to a CSV file called "predictions_submission.csv", which meets the format requirements for "Kaggle".

The first 10 rows are displayed to verify that the recommendations are correctly generated and ready for submission.